In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ==================== 设备配置 ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"使用设备: {device}")

# ==================== 几何参数 ====================
d0 = 0.05  # 内管直径（m）
d1 = 0.13  # 外壳直径（m）
r0 = d0 / 2  # 内管半径
r1 = d1 / 2  # 外壳半径
L_char = d1 - d0  # 特征长度：环形域宽度 (m)

# ==================== 材料参数 ====================
# PCM（石蜡）- 来自论文Table 1
rho_s = 880.0      # 固相密度 (kg/m³)
rho_l = 760.0      # 液相密度 (kg/m³)
cp_s = 2180.0      # 固相定压比热容 (J/(kg·K))
cp_l = 2390.0      # 液相定压比热容 (J/(kg·K))
lambda_s = 0.4     # 固相导热系数 (W/(m·K))
lambda_l = 0.15    # 液相导热系数 (W/(m·K))
mu_l = 0.001       # 液相粘度 (kg/(m·s))
L = 255000.0       # 相变潜热 (J/kg)
Tpc = 316.15       # 相变温度 (K)
DeltaT = 6.0       # 相变温度区间 (K)
alpha = 1.0e-4     # 体膨胀系数 (1/K)
g = 9.81           # 重力加速度 (m/s²)

# 高导热材料（铜）
rho_Cu = 8960.0    # 密度 (kg/m³)
lambda_Cu = 400.0  # 导热系数 (W/(m·K))
cp_Cu = 385.0      # 定压比热容 (J/(kg·K))

# ==================== 计算合理的特征尺度 ====================
def compute_characteristic_scales():
    """基于物理分析计算合理的特征尺度"""
    # 特征速度（基于自然对流）
    DeltaT_scale = 70.0  # 特征温差
    U_char = np.sqrt(g * alpha * DeltaT_scale * L_char)  # 自然对流速度尺度
    
    # 特征时间（基于对流时间）
    t_char = L_char / U_char  # 对流时间
    
    # 特征压力
    p_char = rho_l * U_char**2
    
    print("="*60)
    print("物理合理的特征尺度:")
    print(f"特征长度 L_char = {L_char:.4f} m")
    print(f"特征速度 U_char = {U_char:.4f} m/s")
    print(f"特征时间 t_char = {t_char:.2f} s")
    print(f"特征温差 T_char = {DeltaT_scale:.1f} K")
    print(f"特征压力 p_char = {p_char:.4f} Pa")
    print("="*60)
    
    return {
        'L_char': L_char,
        'U_char': U_char,
        't_char': t_char,
        'T_char': DeltaT_scale,
        'p_char': p_char
    }

char_scales = compute_characteristic_scales()
L_char = char_scales['L_char']
U_char = char_scales['U_char']
t_char = char_scales['t_char']
T_char = char_scales['T_char']
p_char = char_scales['p_char']

# ==================== 拓扑优化参数 ====================
phi_total = 0.3  # 高导热材料体积比约束
case = 1         # 优化目标选择：1=平均温度，2=温度均方差，3=多目标

# ==================== 数值稳定性参数 ====================
eps = 1e-12      # 防止除零的小量

# ==================== 基础网络架构 ====================
class PhysicsInformedNN(nn.Module):
    """物理信息神经网络"""
    
    def __init__(self, input_dim=3, hidden_dim=128, num_layers=6):
        super(PhysicsInformedNN, self).__init__()
        
        # 构建网络
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.Tanh())
        
        for _ in range(num_layers - 2):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        
        layers.append(nn.Linear(hidden_dim, 4))  # 输出: u*, v*, p*, T*
        
        self.net = nn.Sequential(*layers)
        
        # 特征尺度
        self.register_buffer('L_char', torch.tensor(L_char))
        self.register_buffer('U_char', torch.tensor(U_char))
        self.register_buffer('t_char', torch.tensor(t_char))
        self.register_buffer('T_char', torch.tensor(T_char))
        self.register_buffer('p_char', torch.tensor(p_char))
        
        # 参考温度
        self.register_buffer('T0', torch.tensor(290.0))  # 初始温度
        self.register_buffer('Tw_heat', torch.tensor(360.0))  # 储热时内壁温度
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        """初始化网络权重"""
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight, gain=0.5)
                nn.init.zeros_(layer.bias)
    
    def forward(self, x):
        """
        前向传播
        输入: x [batch, 3] - 无量纲坐标 (x*, y*, τ*)
        输出: u, v, p, T
        """
        # 网络输出（无量纲）
        out = self.net(x)
        
        # 转换为有量纲量
        u = out[:, 0:1] * self.U_char  # u = u* * U_char
        v = out[:, 1:2] * self.U_char  # v = v* * U_char
        p = out[:, 2:3] * self.p_char  # p = p* * p_char
        T = out[:, 3:4] * self.T_char + self.T0  # T = T* * T_char + T0
        
        return u, v, p, T

# ==================== 拓扑网络 ====================
class TopologyNetwork(nn.Module):
    """拓扑设计网络"""
    
    def __init__(self, hidden_dim=64, num_layers=4):
        super(TopologyNetwork, self).__init__()
        
        # 构建网络
        layers = []
        layers.append(nn.Linear(2, hidden_dim))
        layers.append(nn.Tanh())
        
        for _ in range(num_layers - 2):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        
        layers.append(nn.Linear(hidden_dim, 1))
        layers.append(nn.Sigmoid())  # 输出在[0,1]之间
        
        self.net = nn.Sequential(*layers)
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        """初始化权重"""
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight, gain=0.1)
                if layer.bias is not None:
                    nn.init.constant_(layer.bias, 0.0)
    
    def forward(self, x_space):
        """
        前向传播
        输入: x_space [batch, 2] - 无量纲空间坐标
        输出: rho [batch, 1] - 拓扑设计变量 ∈ [0,1]
        """
        return self.net(x_space)

# ==================== 材料模型 ====================
class MaterialModel:
    """简化材料模型"""
    
    def __init__(self, device='cpu'):
        self.device = device
        
        # PCM材料参数
        self.Tpc = Tpc
        self.DeltaT = DeltaT
        self.L = L
        self.rho_s = rho_s
        self.rho_l = rho_l
        self.cp_s = cp_s
        self.cp_l = cp_l
        self.lambda_s = lambda_s
        self.lambda_l = lambda_l
        self.mu_l = mu_l
        
        # 铜的材料参数
        self.rho_Cu = rho_Cu
        self.lambda_Cu = lambda_Cu
        self.cp_Cu = cp_Cu
        
    def compute_properties(self, T, rho_design):
        """计算混合材料属性"""
        if isinstance(T, torch.Tensor):
            # 计算液相率
            normalized = (T - self.Tpc) / (self.DeltaT / 2)
            phi = 0.5 * (1.0 + torch.tanh(normalized))
            phi = torch.clamp(phi, 0.0, 1.0)
            
            # PCM密度
            rho_pcm = self.rho_s + (self.rho_l - self.rho_s) * phi
            
            # PCM导热系数
            lambda_pcm = self.lambda_s + (self.lambda_l - self.lambda_s) * phi
            
            # PCM有效比热容
            cp_pcm = self.cp_s + (self.cp_l - self.cp_s) * phi
            
            # 潜热部分
            sigma = self.DeltaT / 4.0
            exponent = -((T - self.Tpc) ** 2) / (2 * sigma ** 2)
            gaussian = torch.exp(exponent) / (sigma * np.sqrt(2 * np.pi))
            cp_pcm += self.L * gaussian
            
            # PCM粘度
            mu_pcm = self.mu_l * phi + 1e4 * (1 - phi)
            
            # 确保rho_design在[0,1]范围内
            rho_design = torch.clamp(rho_design, 0.0, 1.0)
            
            # 混合密度
            rho_total = rho_design * self.rho_Cu + (1 - rho_design) * rho_pcm
            
            # 混合导热系数
            lambda_total = rho_design * self.lambda_Cu + (1 - rho_design) * lambda_pcm
            
            # 混合比热容
            cp_total = rho_design * self.cp_Cu + (1 - rho_design) * cp_pcm
            
            # 混合粘度
            mu_total = rho_design * 1e4 + (1 - rho_design) * mu_pcm
            
        else:
            # NumPy版本（用于验证）
            normalized = (T - self.Tpc) / (self.DeltaT / 2)
            phi = 0.5 * (1.0 + np.tanh(normalized))
            phi = np.clip(phi, 0.0, 1.0)
            
            rho_pcm = self.rho_s + (self.rho_l - self.rho_s) * phi
            lambda_pcm = self.lambda_s + (self.lambda_l - self.lambda_s) * phi
            cp_pcm = self.cp_s + (self.cp_l - self.cp_s) * phi
            
            sigma = self.DeltaT / 4.0
            exponent = -((T - self.Tpc) ** 2) / (2 * sigma ** 2)
            gaussian = np.exp(exponent) / (sigma * np.sqrt(2 * np.pi))
            cp_pcm += self.L * gaussian
            
            mu_pcm = self.mu_l * phi + 1e4 * (1 - phi)
            
            rho_design = np.clip(rho_design, 0.0, 1.0)
            rho_total = rho_design * self.rho_Cu + (1 - rho_design) * rho_pcm
            lambda_total = rho_design * self.lambda_Cu + (1 - rho_design) * lambda_pcm
            cp_total = rho_design * self.cp_Cu + (1 - rho_design) * cp_pcm
            mu_total = rho_design * 1e4 + (1 - rho_design) * mu_pcm
        
        # 计算其他属性
        nu_total = mu_total / (rho_total + eps)
        a_total = lambda_total / (rho_total * cp_total + eps)
        
        return rho_total, lambda_total, cp_total, mu_total, nu_total, a_total, phi

# ==================== 完整的TopoPINN模型 ====================
class TopoPINN(nn.Module):
    """完整的拓扑优化PINN模型"""
    
    def __init__(self, hidden_dim=128, num_layers=6):
        super(TopoPINN, self).__init__()
        
        # 物理场网络
        self.physics_nn = PhysicsInformedNN(
            input_dim=3, 
            hidden_dim=hidden_dim, 
            num_layers=num_layers
        )
        
        # 拓扑网络
        self.topology_nn = TopologyNetwork(
            hidden_dim=64,
            num_layers=4
        )
        
        # 材料模型
        self.material = MaterialModel()
        
        # 物理参数
        self.register_buffer('g_val', torch.tensor(g))
        self.register_buffer('beta', torch.tensor(alpha))
        self.register_buffer('Tpc', torch.tensor(Tpc))
    
    def to(self, device=None, dtype=None):
        """重写to方法"""
        self = super().to(device, dtype)
        if device is not None:
            self.material.device = device
        return self
    
    def forward(self, x):
        """
        前向传播
        输入: x [batch, 3] - 无量纲坐标 (x*, y*, τ*)
        输出: u, v, p, T, rho
        """
        # 提取空间坐标
        x_space = x[:, 0:2]
        
        # 物理场预测
        u, v, p, T = self.physics_nn(x)
        
        # 拓扑设计变量
        rho = self.topology_nn(x_space)
        
        return u, v, p, T, rho
    
    def compute_pde_residuals(self, x):
        """
        计算PDE残差 - 简化版本
        """
        # 确保输入需要梯度
        if not x.requires_grad:
            x = x.clone().requires_grad_(True)
        
        # 前向传播
        u, v, p, T, rho = self(x)
        
        # 计算材料属性
        rho_total, lambda_total, cp_total, mu_total, nu_total, a_total, phi = self.material.compute_properties(T, rho)
        
        # ========== 计算梯度 ==========
        # 分别计算每个输出的梯度
        outputs = [u, v, p, T]
        gradients = []
        
        for output in outputs:
            # 为每个输出单独计算梯度
            grad_output = torch.ones_like(output)
            grad = torch.autograd.grad(
                output, x, 
                grad_outputs=grad_output,
                create_graph=True, 
                retain_graph=True,
                allow_unused=True
            )[0]
            
            if grad is None:
                grad = torch.zeros_like(x)
            
            gradients.append(grad)
        
        grad_u, grad_v, grad_p, grad_T = gradients
        
        # 转换为实际导数
        dudx = grad_u[:, 0:1] / self.physics_nn.L_char
        dudy = grad_u[:, 1:2] / self.physics_nn.L_char
        dudt = grad_u[:, 2:3] / self.physics_nn.t_char
        
        dvdx = grad_v[:, 0:1] / self.physics_nn.L_char
        dvdy = grad_v[:, 1:2] / self.physics_nn.L_char
        dvdt = grad_v[:, 2:3] / self.physics_nn.t_char
        
        dpdx = grad_p[:, 0:1] / self.physics_nn.L_char
        dpdy = grad_p[:, 1:2] / self.physics_nn.L_char
        
        dTdx = grad_T[:, 0:1] / self.physics_nn.L_char
        dTdy = grad_T[:, 1:2] / self.physics_nn.L_char
        dTdt = grad_T[:, 2:3] / self.physics_nn.t_char
        
        # ========== 质量守恒方程 ==========
        # ∇·u = 0 (不可压缩)
        mass_residual = dudx + dvdy
        
        # ========== 动量守恒方程 ==========
        # x方向: ∂u/∂t + u·∇u = -1/ρ ∂p/∂x + ν∇²u
        conv_u = u * dudx + v * dudy
        pressure_u = dpdx / (rho_total + eps)
        viscous_u = nu_total * (dudx + dvdy)  # 简化处理
        
        mom_x_residual = dudt + conv_u + pressure_u - viscous_u
        
        # y方向: ∂v/∂t + u·∇v = -1/ρ ∂p/∂y + ν∇²v + gβ(T - T_ref)
        conv_v = u * dvdx + v * dvdy
        pressure_v = dpdy / (rho_total + eps)
        viscous_v = nu_total * (dudx + dvdy)  # 简化处理
        buoyancy = self.beta * self.g_val * (T - self.Tpc)
        
        mom_y_residual = dvdt + conv_v + pressure_v - viscous_v - buoyancy
        
        # ========== 能量守恒方程 ==========
        # ∂T/∂t + u·∇T = a∇²T
        conv_T = u * dTdx + v * dTdy
        diffusion_T = a_total * (dTdx + dTdy)  # 简化处理
        
        energy_residual = dTdt + conv_T - diffusion_T
        
        return {
            'mass': mass_residual,
            'mom_x': mom_x_residual,
            'mom_y': mom_y_residual,
            'energy': energy_residual,
            'T': T,
            'rho_design': rho,
            'phi': phi
        }

# ==================== 采样器 ====================
class Sampler:
    """采样器"""
    
    def __init__(self, r0, r1, t_char):
        self.r0 = r0
        self.r1 = r1
        self.t_char = t_char
    
    def sample_domain(self, N, time_dependent=True):
        """采样计算域内部点"""
        # 空间采样（环形域）
        r = np.sqrt(np.random.uniform(r0**2, r1**2, N))
        theta = np.random.uniform(0, 2*np.pi, N)
        
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        
        # 时间采样
        if time_dependent:
            t = np.random.uniform(0, self.t_char, N)
        else:
            t = np.full(N, 0.5 * self.t_char)
        
        # 无量纲化
        x_star = x / L_char
        y_star = y / L_char
        tau_star = t / self.t_char
        
        points = np.column_stack([x_star, y_star, tau_star])
        
        return torch.tensor(points, dtype=torch.float32)
    
    def sample_boundary(self, N, boundary='inner', time_dependent=True):
        """采样边界点"""
        if boundary == 'inner':
            r = self.r0
        else:
            r = self.r1
        
        theta = np.random.uniform(0, 2*np.pi, N)
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        
        if time_dependent:
            t = np.random.uniform(0, self.t_char, N)
        else:
            t = np.zeros(N)
        
        # 无量纲化
        x_star = x / L_char
        y_star = y / L_char
        tau_star = t / self.t_char
        
        points = np.column_stack([x_star, y_star, tau_star])
        
        return torch.tensor(points, dtype=torch.float32)
    
    def sample_initial(self, N):
        """采样初始条件点"""
        r = np.sqrt(np.random.uniform(r0**2, r1**2, N))
        theta = np.random.uniform(0, 2*np.pi, N)
        
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        t = np.zeros(N)
        
        # 无量纲化
        x_star = x / L_char
        y_star = y / L_char
        tau_star = t / self.t_char
        
        points = np.column_stack([x_star, y_star, tau_star])
        
        return torch.tensor(points, dtype=torch.float32)

# ==================== 损失计算器 ====================
class LossCalculator:
    """损失计算器"""
    
    def __init__(self, model):
        self.model = model
        self.sampler = Sampler(r0, r1, t_char)
        
        # 权重
        self.weights = {
            'pde': 1.0,
            'ic': 10.0,
            'bc_inner': 100.0,
            'bc_outer': 10.0,
            'volume': 100.0,
            'objective': 1.0
        }
        
        # 当前体积分数
        self.current_volume = 0.0
    
    def compute_total_loss(self, stage='pretrain'):
        """计算总损失"""
        # 采样点
        N_domain = 200
        N_boundary = 50
        N_initial = 50
        
        x_domain = self.sampler.sample_domain(N_domain).to(device)
        x_inner = self.sampler.sample_boundary(N_boundary, 'inner').to(device)
        x_outer = self.sampler.sample_boundary(N_boundary, 'outer').to(device)
        x_initial = self.sampler.sample_initial(N_initial).to(device)
        
        # 计算各项损失
        losses = {}
        
        # PDE损失
        losses['pde'] = self.compute_pde_loss(x_domain)
        
        # 初始条件损失
        losses['ic'] = self.compute_ic_loss(x_initial)
        
        # 边界条件损失
        losses['bc_inner'] = self.compute_bc_loss(x_inner, boundary='inner')
        losses['bc_outer'] = self.compute_bc_loss(x_outer, boundary='outer')
        
        # 体积约束损失
        losses['volume'] = self.compute_volume_loss(x_domain)
        
        # 优化目标损失
        losses['objective'] = self.compute_objective_loss(x_domain)
        
        # 计算加权总损失
        total_loss = torch.tensor(0.0).to(device)
        for key, loss in losses.items():
            total_loss += self.weights[key] * loss
        
        return total_loss, losses
    
    def compute_pde_loss(self, x):
        """计算PDE残差损失"""
        residuals = self.model.compute_pde_residuals(x)
        
        # 各项残差的均方根
        mass_loss = torch.mean(residuals['mass'] ** 2)
        mom_x_loss = torch.mean(residuals['mom_x'] ** 2)
        mom_y_loss = torch.mean(residuals['mom_y'] ** 2)
        energy_loss = torch.mean(residuals['energy'] ** 2)
        
        return mass_loss + mom_x_loss + mom_y_loss + energy_loss
    
    def compute_ic_loss(self, x):
        """计算初始条件损失"""
        u, v, p, T, rho = self.model(x)
        
        # 初始温度应为T0，速度应为0
        T0 = self.model.physics_nn.T0
        T_loss = torch.mean((T - T0) ** 2)
        u_loss = torch.mean(u ** 2)
        v_loss = torch.mean(v ** 2)
        
        return T_loss + 0.1 * (u_loss + v_loss)
    
    def compute_bc_loss(self, x, boundary='inner'):
        """计算边界条件损失"""
        u, v, p, T, rho = self.model(x)
        
        if boundary == 'inner':
            # 内壁：恒温边界
            Tw = self.model.physics_nn.Tw_heat
            T_loss = torch.mean((T - Tw) ** 2)
            
            # 无滑移边界条件
            u_loss = torch.mean(u ** 2)
            v_loss = torch.mean(v ** 2)
            
            return T_loss + 0.1 * (u_loss + v_loss)
        else:
            # 外壁：绝热边界（简化处理）
            return torch.tensor(0.0).to(device)
    
    def compute_volume_loss(self, x):
        """计算体积约束损失"""
        x_space = x[:, 0:2]
        rho = self.model.topology_nn(x_space)
        
        # 计算平均体积分数
        volume = torch.mean(rho)
        self.current_volume = volume.item()
        
        # 体积约束损失
        violation = volume - phi_total
        return torch.mean(violation ** 2)
    
    def compute_objective_loss(self, x):
        """计算优化目标损失"""
        u, v, p, T, rho = self.model(x)
        
        if case == 1:
            # 最小化平均温度
            T_avg = torch.mean(T)
            return (T_avg - 320.0) ** 2
        elif case == 2:
            # 最小化温度方差
            T_avg = torch.mean(T)
            T_var = torch.mean((T - T_avg) ** 2)
            return T_var
        else:
            # 多目标优化
            T_avg = torch.mean(T)
            T_var = torch.mean((T - T_avg) ** 2)
            
            # 计算温度梯度
            x = x.clone().requires_grad_(True)
            T = self.model(x)[3]
            
            grad_T = torch.autograd.grad(T, x, 
                                         grad_outputs=torch.ones_like(T),
                                         create_graph=True, allow_unused=True)[0]
            
            if grad_T is None:
                grad_T_sq = torch.tensor(0.0).to(device)
            else:
                dTdx = grad_T[:, 0:1] / self.model.physics_nn.L_char
                dTdy = grad_T[:, 1:2] / self.model.physics_nn.L_char
                grad_T_sq = torch.mean(dTdx**2 + dTdy**2)
            
            return 0.5 * (T_avg - 320.0) ** 2 + 0.3 * T_var + 0.2 * grad_T_sq

# ==================== 训练器 ====================
class Trainer:
    """训练器"""
    
    def __init__(self, model):
        self.model = model
        self.loss_calculator = LossCalculator(model)
        
        # 训练阶段
        self.stages = [
            {'name': 'pretrain', 'epochs': 10, 'lr': 1e-3, 'desc': '预训练物理场'},
            {'name': 'joint', 'epochs': 20, 'lr': 5e-4, 'desc': '联合训练'},
        ]
        
        # 训练历史
        self.history = {
            'total_loss': [],
            'volume': [],
            'stage': []
        }
        
        # 创建保存目录
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.save_dir = f"./results/case{case}_{timestamp}"
        os.makedirs(self.save_dir, exist_ok=True)
    
    def train(self):
        """训练模型"""
        print("="*60)
        print("开始训练拓扑优化PINN")
        print("="*60)
        
        total_epochs = 0
        
        for stage_idx, stage_config in enumerate(self.stages):
            stage_name = stage_config['name']
            epochs = stage_config['epochs']
            lr = stage_config['lr']
            
            print(f"\n{'='*60}")
            print(f"阶段 {stage_idx+1}: {stage_config['desc']}")
            print(f"轮次: {epochs}, 学习率: {lr}")
            print(f"{'='*60}")
            
            # 设置优化器
            if stage_name == 'pretrain':
                # 预训练阶段：只训练物理场网络
                params = list(self.model.physics_nn.parameters())
                for param in self.model.topology_nn.parameters():
                    param.requires_grad = False
            else:
                # 联合训练阶段：训练所有参数
                params = self.model.parameters()
                for param in self.model.parameters():
                    param.requires_grad = True
            
            optimizer = optim.Adam(params, lr=lr)
            scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
            
            # 阶段训练
            for epoch in range(epochs):
                total_epochs += 1
                
                try:
                    # 计算损失
                    total_loss, losses = self.loss_calculator.compute_total_loss(stage=stage_name)
                    
                    # 反向传播
                    optimizer.zero_grad()
                    total_loss.backward()
                    
                    # 梯度裁剪
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    
                    # 优化
                    optimizer.step()
                    scheduler.step(total_loss)
                    
                    # 记录历史
                    self.history['total_loss'].append(total_loss.item())
                    self.history['volume'].append(self.loss_calculator.current_volume)
                    self.history['stage'].append(stage_idx)
                    
                    # 输出进度
                    if (epoch + 1) % 2 == 0:
                        print(f"Epoch {total_epochs:4d} | Loss: {total_loss.item():.4e} | "
                              f"Volume: {self.loss_calculator.current_volume:.4f} | "
                              f"PDE: {losses['pde'].item():.2e} | "
                              f"BC: {losses['bc_inner'].item():.2e}")
                
                except Exception as e:
                    print(f"训练出错: {e}")
                    break
        
        print("\n训练完成!")
        self.save_results()
    
    def save_results(self):
        """保存结果"""
        # 保存模型
        torch.save(self.model.state_dict(), f"{self.save_dir}/model_final.pth")
        
        # 保存训练历史
        np.savez(f"{self.save_dir}/training_history.npz",
                 total_loss=self.history['total_loss'],
                 volume=self.history['volume'],
                 stage=self.history['stage'])
        
        # 绘制训练曲线
        self.plot_training_history()
        
        print(f"\n所有结果已保存到: {self.save_dir}")
    
    def plot_training_history(self):
        """绘制训练历史曲线"""
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # 总损失
        axes[0].semilogy(self.history['total_loss'])
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Total Loss')
        axes[0].set_title('Total Loss History')
        axes[0].grid(True, alpha=0.3)
        
        # 体积分数
        axes[1].plot(self.history['volume'])
        axes[1].axhline(y=phi_total, color='r', linestyle='--', label=f'Target: {phi_total}')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Volume Fraction')
        axes[1].set_title('Volume Constraint')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{self.save_dir}/training_history.png", dpi=300)
        plt.close()

# ==================== 验证函数 ====================
def validate_model(model):
    """验证模型"""
    print("\n" + "="*60)
    print("模型验证")
    print("="*60)
    
    sampler = Sampler(r0, r1, t_char)
    
    # 采样验证点
    x_val = sampler.sample_domain(50).to(device)
    x_inner = sampler.sample_boundary(20, 'inner').to(device)
    x_initial = sampler.sample_initial(20).to(device)
    
    with torch.no_grad():
        # 验证边界条件
        u_inner, v_inner, p_inner, T_inner, rho_inner = model(x_inner)
        T_error_inner = torch.mean((T_inner - 360.0) ** 2).item()
        print(f"内壁温度误差: {T_error_inner:.2e}")
        
        # 验证初始条件
        u_initial, v_initial, p_initial, T_initial, rho_initial = model(x_initial)
        T_error_initial = torch.mean((T_initial - 290.0) ** 2).item()
        print(f"初始温度误差: {T_error_initial:.2e}")
        
        # 验证体积约束
        x_space = x_val[:, 0:2]
        rho = model.topology_nn(x_space)
        volume = torch.mean(rho).item()
        print(f"体积分数: {volume:.4f} (目标: {phi_total})")
        
        # 验证物理合理性
        print(f"\n物理量范围:")
        print(f"  温度范围: {T_initial.min().item():.1f}K - {T_initial.max().item():.1f}K")
        print(f"  速度范围: u[{u_initial.min().item():.2e}, {u_initial.max().item():.2e}] m/s")
        print(f"  拓扑变量: [{rho.min().item():.3f}, {rho.max().item():.3f}]")
    
    # 计算PDE残差
    print("\n计算PDE残差...")
    x_val.requires_grad_(True)
    residuals = model.compute_pde_residuals(x_val)
    
    print("PDE残差:")
    print(f"  质量守恒: {torch.mean(residuals['mass']**2).item():.2e}")
    print(f"  x动量守恒: {torch.mean(residuals['mom_x']**2).item():.2e}")
    print(f"  y动量守恒: {torch.mean(residuals['mom_y']**2).item():.2e}")
    print(f"  能量守恒: {torch.mean(residuals['energy']**2).item():.2e}")
    
    print("="*60)

# ==================== 主程序 ====================
if __name__ == "__main__":
    print("拓扑优化PINN - 相变储热系统（简化版）")
    print("="*60)
    
    # 1. 初始化模型
    print("初始化模型...")
    model = TopoPINN(hidden_dim=128, num_layers=6).to(device)
    
    # 2. 验证初始模型
    validate_model(model)
    
    # 3. 训练模型
    trainer = Trainer(model)
    trainer.train()
    
    # 4. 验证训练后模型
    validate_model(model)
    
    # 5. 保存和可视化结果
    print("\n保存和可视化结果...")
    
    # 创建网格进行可视化
    r = np.linspace(r0, r1, 20)
    theta = np.linspace(0, 2*np.pi, 20)
    R, Theta = np.meshgrid(r, theta)
    X = R * np.cos(Theta)
    Y = R * np.sin(Theta)
    
    # 准备输入
    x_flat = X.flatten()[:, np.newaxis] / L_char
    y_flat = Y.flatten()[:, np.newaxis] / L_char
    t_flat = np.full_like(x_flat, 0.5)
    
    x_input = np.hstack([x_flat, y_flat, t_flat])
    x_tensor = torch.tensor(x_input, dtype=torch.float32).to(device)
    
    with torch.no_grad():
        u, v, p, T, rho = model(x_tensor)
    
    # 重塑为网格
    T_grid = T.cpu().numpy().reshape(20, 20)
    rho_grid = rho.cpu().numpy().reshape(20, 20)
    u_mag = torch.sqrt(u**2 + v**2).cpu().numpy().reshape(20, 20)
    
    # 绘图
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 温度场
    im1 = axes[0].contourf(X, Y, T_grid, levels=20, cmap='jet', vmin=290, vmax=360)
    axes[0].set_title('Temperature Distribution')
    axes[0].set_xlabel('x (m)')
    axes[0].set_ylabel('y (m)')
    axes[0].axis('equal')
    plt.colorbar(im1, ax=axes[0])
    
    # 拓扑结构
    im2 = axes[1].contourf(X, Y, rho_grid, levels=20, cmap='binary', vmin=0, vmax=1)
    axes[1].set_title('Topology Distribution')
    axes[1].set_xlabel('x (m)')
    axes[1].set_ylabel('y (m)')
    axes[1].axis('equal')
    plt.colorbar(im2, ax=axes[1])
    
    # 速度场
    im3 = axes[2].contourf(X, Y, u_mag, levels=20, cmap='plasma')
    axes[2].set_title('Velocity Magnitude')
    axes[2].set_xlabel('x (m)')
    axes[2].set_ylabel('y (m)')
    axes[2].axis('equal')
    plt.colorbar(im3, ax=axes[2])
    
    plt.tight_layout()
    plt.savefig(f"{trainer.save_dir}/final_results.png", dpi=300)
    plt.close()
    
    print(f"\n最终结果图已保存到: {trainer.save_dir}/final_results.png")
    print("\n" + "="*60)
    print("程序执行完成!")
    print(f"结果保存目录: {trainer.save_dir}")
    print("="*60)

使用设备: cuda
物理合理的特征尺度:
特征长度 L_char = 0.0800 m
特征速度 U_char = 0.0741 m/s
特征时间 t_char = 1.08 s
特征温差 T_char = 70.0 K
特征压力 p_char = 4.1751 Pa
拓扑优化PINN - 相变储热系统（简化版）
初始化模型...

模型验证
内壁温度误差: 4.90e+03
初始温度误差: 7.84e-03
体积分数: 0.5000 (目标: 0.3)

物理量范围:
  温度范围: 289.8K - 290.2K
  速度范围: u[-2.38e-04, 2.09e-04] m/s
  拓扑变量: [0.500, 0.500]

计算PDE残差...
PDE残差:
  质量守恒: 8.35e-06
  x动量守恒: 3.46e-05
  y动量守恒: 3.92e-04
  能量守恒: 8.75e-04
开始训练拓扑优化PINN

阶段 1: 预训练物理场
轮次: 10, 学习率: 0.001
Epoch    2 | Loss: 4.6724e+05 | Volume: 0.5000 | PDE: 1.94e+00 | BC: 4.66e+03
Epoch    4 | Loss: 3.9837e+05 | Volume: 0.5000 | PDE: 3.61e+01 | BC: 3.98e+03
Epoch    6 | Loss: 2.7372e+05 | Volume: 0.5000 | PDE: 2.79e+02 | BC: 2.72e+03
Epoch    8 | Loss: 1.0197e+05 | Volume: 0.5000 | PDE: 1.55e+03 | BC: 9.66e+02
Epoch   10 | Loss: 8.7605e+04 | Volume: 0.5000 | PDE: 6.49e+03 | BC: 6.54e+02

阶段 2: 联合训练
轮次: 20, 学习率: 0.0005
Epoch   12 | Loss: 9.0583e+04 | Volume: 0.4997 | PDE: 5.41e+03 | BC: 6.94e+02
Epoch   14 | Loss: 5.2227e+04 | Volume: 0.49